# Gold layer — mBART metadata / extraction

Pipeline before model use:
upload pdf -> extract to txt -> clean txt -> preprocess for nlp -> metadata extraction -> classification -> output metadata


In [251]:
# Load the necessary libraries
import json
import re
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
import torch
from transformers import pipeline
from datetime import datetime

In [252]:
# Configuration
INPUT_JSON = "../../../Data/silver/doc_01_silver_nlp.json"
OUTPUT_DIR = "../../../Data/gold/MBART/"
OUTPUT_FILE = "mbart_metadata_output.json"

DOC_ID_COLUMN = "document_id"
RAW_TEXT_COLUMN = "raw_text"
NLP_TEXT_COLUMN = "nlp_text"
TEXT_COLUMN = "cleaned_text"
ENTITIES_COLUMN = "entities"

ZERO_SHOT_MODEL = "facebook/bart-large-mnli"

RESEARCH_GROUPS = [
    "Aquaculture in Delta Areas",
    "Assetmanagement",
    "Biobased Bouwen",
    "Building with Nature",
    "Data Science",
    "Delta Power",
    "Excellence and Innovation in Education",
    "HZ Kenniscentrum Kusttoerisme",
    "HZ Kenniscentrum Ondernemen en Innoiveren",
    "HZ Kenniscentrum Zeeuwse Samenleving",
    "HZ Nexus",
    "Healthy Region",
    "Kunst Cultuur Transitie",
    "Marine Biobased Chemie",
    "Ouderenzorg",
    "Resilient Deltas",
    "Supply Chain Innovation",
    "Water Technology",
]

RESEARCH_GROUP_CONFIDENCE_THRESHOLD = 0.20


In [253]:
# Load zero-shot classifier
zero_shot_classifier = pipeline(
    "zero-shot-classification",
    model=ZERO_SHOT_MODEL,
    device=0 if torch.cuda.is_available() else -1
)

print("Loaded zero-shot classifier.")


Loading weights: 100%|██████████| 515/515 [00:00<00:00, 14053.42it/s]


Loaded zero-shot classifier.


In [254]:
# Helpers
SECTION_LABELS = {
    "abstract", "introduction", "background", "discussion", "conclusion",
    "conclusions", "results", "result", "method", "methods", "methodology",
    "references", "bibliography", "appendix", "summary", "contents",
    "table of contents"
}

NON_PERSON_TERMS = {
    "report", "research", "assignment", "results", "introduction",
    "conclusion", "discussion", "services", "selection", "configuration",
    "installation", "implementation", "contents", "appendix", "appendixes",
    "references", "chapter", "table", "figure", "performance", "qualities",
    "cloud", "computing", "outlook", "calendar", "calander", "subject",
    "portfolio", "development", "professional", "personal"
}

INSTITUTION_PATTERNS = [
    r"\buniversity\b",
    r"\bcollege\b",
    r"\bfaculty\b",
    r"\bschool\b",
    r"\binstitute\b",
    r"\bdepartment\b",
    r"\bapplied sciences\b",
    r"\bhz university\b"
]

MONTH_WORDS = (
    "january|february|march|april|may|june|july|august|september|october|november|december|"
    "jan|feb|mar|apr|jun|jul|aug|sep|sept|oct|nov|dec"
)

DATE_PATTERN = (
    r"\b\d{1,2}(?:st|nd|rd|th)?[/-]\d{1,2}[/-]\d{2,4}\b|"
    r"\b\d{4}-\d{2}-\d{2}\b|"
    rf"\b\d{{1,2}}(?:st|nd|rd|th)?\s+(?:{MONTH_WORDS})\s+\d{{2,4}}\b|"
    rf"\b\d{{1,2}}(?:st|nd|rd|th)?\s+(?:{MONTH_WORDS})\b|"
    rf"\b(?:{MONTH_WORDS})\s+\d{{1,2}}(?:st|nd|rd|th)?,?\s+\d{{2,4}}\b|"
    rf"\b(?:{MONTH_WORDS})\s+\d{{4}}\b|"
    r"\b\d{4}\b"
)

DATE_CONTEXT_POSITIVE = [
    r"\bdate\b",
    r"\bpublication date\b",
    r"\bpublished\b",
    r"\breport date\b",
    r"\breleased\b",
    r"\bissued\b",
    r"\bversion\b",
    r"\bfinal report\b",
    r"\bsubmission date\b",
]

DATE_CONTEXT_NEGATIVE = [
    r"\breferences\b",
    r"\bbibliography\b",
    r"\bworks cited\b",
    r"\bet al\.\b",
    r"\bdoi\b",
    r"https?://",
    r"www\.",
    r"\bjournal\b",
    r"\bvol\.\b",
    r"\bno\.\b",
    r"\bpp\.\b",
]

YEAR_ONLY_RE = re.compile(r"^\d{4}$")
def clean_date_text(date_text: str) -> str:
    date_text = normalize_whitespace(date_text).lower()
    date_text = re.sub(r"(\d{1,2})(st|nd|rd|th)\b", r"\1", date_text)
    return date_text.replace(",", "")

def normalize_whitespace(text: Any) -> str:
    return re.sub(r"\s+", " ", "" if text is None else str(text)).strip()

def extract_front_matter_for_dates(text: str, max_lines: int = 60) -> List[str]:
    """
    Only use early document lines, before abstract/TOC/main body if possible.
    """
    text = str(text or "").replace("\r", "\n")
    raw_lines = [line.strip() for line in text.splitlines() if line.strip()]

    stop_patterns = [
        r"^\s*abstract\s*$",
        r"^\s*table\s+of\s+contents\s*$",
        r"^\s*contents\s*$",
        r"^\s*1(\.0+)?\s+(introduction|background)\b",
        r"^\s*introduction\s*$",
    ]

    kept = []
    for line in raw_lines[:max_lines]:
        if any(re.search(p, line, flags=re.IGNORECASE) for p in stop_patterns):
            break
        kept.append(line)

    return kept[:max_lines]

def line_looks_like_reference(line: str) -> bool:
    s = normalize_whitespace(line)
    if not s:
        return False

    if any(re.search(p, s, flags=re.IGNORECASE) for p in DATE_CONTEXT_NEGATIVE):
        return True

    # citation style patterns
    if re.search(r"[A-Z][A-Za-z'\-]+,\s+[A-Z]\.", s):
        return True
    if re.search(r"\(\s*\d{4}[a-z]?\s*\)", s):
        return True

    return False

def score_date_candidate(date_text: str, line: str, line_index: int) -> float:
    s = normalize_whitespace(line)
    score = 0.0

    # early lines are better
    score += max(0, 10 - line_index) * 0.4

    # positive date context
    for p in DATE_CONTEXT_POSITIVE:
        if re.search(p, s, flags=re.IGNORECASE):
            score += 2.0

    # negative context
    for p in DATE_CONTEXT_NEGATIVE:
        if re.search(p, s, flags=re.IGNORECASE):
            score -= 4.0

    # avoid lines with many dates
    num_dates = len(re.findall(DATE_PATTERN, s, flags=re.IGNORECASE))
    if num_dates > 2:
        score -= 2.0

    # standalone line containing mostly a date is often good on title pages
    if len(s) < 40 and date_text in s:
        score += 1.5

    # year-only dates are weak unless context is strong
    if YEAR_ONLY_RE.fullmatch(date_text):
        score -= 2.0

    return score

def extract_document_dates(text: str) -> Dict[str, Any]:
    lines = extract_front_matter_for_dates(text, max_lines=60)

    candidates = []
    for idx, line in enumerate(lines):
        s = normalize_whitespace(line)
        if not s:
            continue
        if line_looks_like_reference(s):
            continue

        matches = re.findall(DATE_PATTERN, s, flags=re.IGNORECASE)
        for match in matches:
            date_text = normalize_whitespace(match)
            if not date_text:
                continue

            score = score_date_candidate(date_text, s, idx)
            candidates.append({
                "date": date_text,
                "line": s,
                "line_index": idx,
                "score": score
            })

    # dedupe by date, keep highest-scoring occurrence
    best_by_date = {}
    for item in candidates:
        key = item["date"].lower()
        if key not in best_by_date or item["score"] > best_by_date[key]["score"]:
            best_by_date[key] = item

    ranked = sorted(
        best_by_date.values(),
        key=lambda x: (-x["score"], x["line_index"])
    )

    valid = [x for x in ranked if x["score"] >= 1.0]

    parsed_valid = []
    for item in valid:
        parsed = parse_date_to_sortable(item["date"], context_line=item["line"])
        if parsed is not None:
            parsed_valid.append({
                **item,
                "parsed_date": parsed
            })

    parsed_valid = sorted(parsed_valid, key=lambda x: x["parsed_date"])

    if len(parsed_valid) >= 2:
        start_date = parsed_valid[0]["date"]
        end_date = parsed_valid[-1]["date"]
    elif len(parsed_valid) == 1:
        start_date = parsed_valid[0]["date"]
        end_date = ""
    else:
        start_date = ""
        end_date = ""

    return {
        "start_date": start_date,
        "end_date": end_date,
        "dates_found": [x["date"] for x in parsed_valid[:5]],
        "date_candidates": ranked[:10],
        "date_confidence": round(valid[0]["score"], 3) if valid else None
    }
    
AUTHOR_PREFIX_PATTERNS = [
    r"^(author|authors|written by|student|students|name|contributors?)\s*[:\-]?\s+",
    r"^by\s+[A-Z]"
]

PERSON_CONNECTORS = {"de", "van", "von", "der", "den", "ten", "ter", "la", "le", "du", "di", "op"}


def safe_string(value: Any) -> str:
    return "" if value is None else str(value)

def dedupe_keep_order(items: List[str]) -> List[str]:
    seen = set()
    out = []
    for item in items:
        clean = normalize_whitespace(item)
        key = clean.lower()
        if clean and key not in seen:
            seen.add(key)
            out.append(clean)
    return out

In [255]:
# Front matter helpers
def extract_front_matter(text: str, max_chars: int = 15000) -> str:
    text = safe_string(text)
    if not text:
        return ""

    text = text.replace("\r", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)

    toc_match = re.search(r"\btable\s+of\s+contents\b", text, flags=re.IGNORECASE)
    if toc_match:
        return text[:toc_match.start()].strip()

    abstract_match = re.search(r"\babstract\b", text, flags=re.IGNORECASE)
    if abstract_match and abstract_match.start() > 0:
        return text[:abstract_match.start()].strip()

    return text[:max_chars].strip()


def get_front_lines_raw(text: str, max_chars: int = 15000, max_lines: int = 80) -> List[str]:
    front = extract_front_matter(text, max_chars=max_chars)
    lines = []
    for line in front.splitlines():
        raw = safe_string(line).strip()
        if raw:
            lines.append(raw)
    return lines[:max_lines]


def split_joined_front_line(line: str) -> List[str]:
    s = normalize_whitespace(line)
    if not s:
        return []

    # Never split institution/org lines — they look like names but aren't
    if any(re.search(p, s, flags=re.IGNORECASE) for p in INSTITUTION_PATTERNS):
        return [s]

    parts: List[str] = []

    author_match = re.search(
        r"([A-Z][a-z]+(?:\s+(?:de|den|der|van|von|ten|ter|op|la|le|du|di))?(?:\s+[A-Z][a-z]+)+"
        r"(?:\s*,\s*[A-Z][a-z]+(?:\s+(?:de|den|der|van|von|ten|ter|op|la|le|du|di))?(?:\s+[A-Z][a-z]+)+)*)$",
        s
    )

    author_part = ""
    if author_match:
        author_part = author_match.group(1).strip()
        s = s[:author_match.start()].strip()

    code_match = re.search(r"\([A-Z]{2,6}\d{4,}[A-Z0-9]*\)", s)
    if code_match:
        left = s[:code_match.end()].strip()
        right = s[code_match.end():].strip()
        if left:
            parts.append(left)
        if right:
            parts.append(right)
    else:
        parts.append(s)

    final_parts: List[str] = []
    for part in parts:
        part = normalize_whitespace(part)
        if not part:
            continue

        caps_match = re.search(r"\b([A-Z][A-Z\s&\-]{5,})\b", part)
        if caps_match and caps_match.group(1).strip() != part.strip():
            before = normalize_whitespace(part[:caps_match.start()])
            caps = normalize_whitespace(caps_match.group(1))
            after = normalize_whitespace(part[caps_match.end():])

            if before:
                final_parts.append(before)
            if caps:
                final_parts.append(caps)
            if after:
                final_parts.append(after)
        else:
            final_parts.append(part)

    if author_part:
        final_parts.append(author_part)

    return [p for p in final_parts if p]

def get_front_lines(text: str, max_chars: int = 15000, max_lines: int = 80) -> List[str]:
    raw_lines = get_front_lines_raw(text, max_chars=max_chars, max_lines=max_lines)

    processed: List[str] = []
    for line in raw_lines:
        split_parts = split_joined_front_line(line)
        if split_parts:
            processed.extend(split_parts)
        else:
            processed.append(normalize_whitespace(line))

    return [normalize_whitespace(x) for x in processed if normalize_whitespace(x)][:max_lines]

def classify_front_line(line: str) -> str:
    s = normalize_whitespace(line)

    if not s:
        return "noise"
    if is_table_of_contents_line(s):
        return "toc"
    if is_likely_org_line(s):
        return "organization"
    if extract_contributors_from_line(s):
        return "author"
    if re.search(DATE_PATTERN, s, flags=re.IGNORECASE):
        return "date"
    if s.lower() in SECTION_LABELS:
        return "section"
    if len(s.split()) <= 12 and not looks_like_reference(s):
        return "title_like"
    return "noise"

In [256]:
# Contributor helpers
def normalize_person_casing(name: str) -> str:
    parts = []
    for p in normalize_whitespace(name).split():
        if p.lower() in PERSON_CONNECTORS:
            parts.append(p.lower())
        elif p.isupper():
            parts.append(p.capitalize())
        else:
            parts.append(p)
    return " ".join(parts)

def is_valid_person_name(name: str) -> bool:
    name = normalize_whitespace(name)

    if len(name) < 5:
        return False

    parts = name.split()
    if len(parts) < 2 or len(parts) > 4:
        return False

    if any(ch.isdigit() for ch in name):
        return False
    if re.search(r"[•()\[\]{}:;\\/]", name):
        return False

    lowered_parts = [p.lower() for p in parts if p.lower() not in PERSON_CONNECTORS]

    # Reject obvious non-person vocabulary
    if any(p in NON_PERSON_TERMS for p in lowered_parts):
        return False

    # Require each non-connector part to look like a human name token
    proper_tokens = 0
    for p in parts:
        if p.lower() in PERSON_CONNECTORS:
            continue
        if re.fullmatch(r"[A-Z][a-zà-ÿ'\-]{1,20}", p):
            proper_tokens += 1
        else:
            return False

    if proper_tokens < 2:
        return False

    return True

def extract_person_entities_from_row(row: pd.Series) -> List[str]:
    entities = row.get(ENTITIES_COLUMN, [])
    found = []

    if isinstance(entities, list):
        for ent in entities:
            if isinstance(ent, dict) and ent.get("label") == "PERSON":
                candidate = normalize_whitespace(ent.get("text", ""))
                if is_valid_person_name(candidate):
                    found.append(candidate)

    return dedupe_keep_order(found)


def extract_contributors_from_line(line: str) -> List[str]:
    s = normalize_whitespace(line)
    if not s:
        return []

    # Skip lines that are institution/org names — they are never contributor lines
    if any(re.search(p, s, flags=re.IGNORECASE) for p in INSTITUTION_PATTERNS):
        return []

    for pattern in AUTHOR_PREFIX_PATTERNS:
        s = re.sub(pattern, "", s, flags=re.IGNORECASE)

    parts = [normalize_whitespace(p) for p in re.split(r",| and | & |;", s) if normalize_whitespace(p)]
    normalized_parts = [normalize_person_casing(p) for p in parts]
    names = [p for p in normalized_parts if is_valid_person_name(p)]
    return dedupe_keep_order(names)


def extract_contributors(row: pd.Series, max_authors: int = 8) -> List[str]:
    text = safe_string(row.get(NLP_TEXT_COLUMN, "")) or safe_string(row.get(TEXT_COLUMN, ""))
    lines = get_title_page_lines(text, max_lines=20)

    ner_people = set(extract_person_entities_from_row(row))
    found = []

    for line in lines:
        if is_likely_org_line(line) or is_table_of_contents_line(line):
            continue

        names = extract_contributors_from_line(line)
        for n in names:
            found.append(n)

    for person in ner_people:
        if person not in found:
            # optional: only add if person occurs in title-page lines
            if any(person.lower() in line.lower() for line in lines):
                found.append(person)

    return dedupe_keep_order(found)[:max_authors]

In [257]:
# Title helpers
def get_title_page_lines(text: str, max_lines: int = 20) -> List[str]:
    text = safe_string(text).replace("\r", "\n")
    raw_lines = [line.strip() for line in text.splitlines() if line.strip()]

    stop_patterns = [
        r"^\s*contents\s*$",
        r"^\s*table\s+of\s+contents\s*$",
        r"^\s*abstract\s*$",
        r"^\s*\d+(\.\d+)*\s+[A-Za-z]",
    ]

    kept = []
    for line in raw_lines[:max_lines]:
        if any(re.search(p, line, flags=re.IGNORECASE) for p in stop_patterns):
            break
        kept.append(line)

    return kept

def is_table_of_contents_line(line: str) -> bool:
    s = normalize_whitespace(line)

    if "table of contents" in s.lower():
        return True
    if re.search(r"\.{4,}\s*\d+$", s):
        return True
    if re.search(r"\.{4,}", s):
        return True
    if re.match(r"^\d+(\.\d+)*\s+.+\s+\d+$", s.lower()):
        return True
    if re.search(r"\b\d+\.\d+\b", s) and len(s) < 120:
        return True

    return False


def looks_like_reference(sentence: str) -> bool:
    s = normalize_whitespace(sentence)

    if re.search(r"\bet al\.", s, flags=re.IGNORECASE):
        return True
    if re.search(r"\bdoi\b", s, flags=re.IGNORECASE):
        return True
    if re.search(r"https?://|www\.", s):
        return True
    if re.search(r"\breferences\b|\bbibliography\b", s, flags=re.IGNORECASE):
        return True
    if re.search(r"\(\s*[A-Z][A-Za-z].*\d{4}.*\)", s):
        return True

    return False


def is_date_like_line(line: str) -> bool:
    s = normalize_whitespace(line)

    if not s:
        return False
    if re.fullmatch(r"\d{1,2}[/-]\d{1,2}[/-]\d{2,4}", s):
        return True
    if re.fullmatch(r"\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\s*[-–]\s*\d{1,2}[/-]\d{1,2}[/-]\d{2,4}", s):
        return True
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", s):
        return True
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}\s*[-–]\s*\d{4}-\d{2}-\d{2}", s):
        return True

    return False


def is_title_noise(line: str) -> bool:
    s = normalize_whitespace(line)

    if not s:
        return True
    if s.lower() in SECTION_LABELS:
        return True
    if is_table_of_contents_line(s):
        return True
    if looks_like_reference(s):
        return True
    if is_date_like_line(s):
        return True
    if re.fullmatch(r"\d+\s*\|\s*p\s*a\s*g\s*e", s, flags=re.IGNORECASE):
        return True
    if re.fullmatch(
        r"\d+(\.\d+)*\s*(introduction|background|methods?|results?|discussion|conclusions?|references|appendix)",
        s,
        flags=re.IGNORECASE
    ):
        return True
    return False


def is_likely_org_line(line: str) -> bool:
    s = normalize_whitespace(line)
    return any(re.search(p, s, flags=re.IGNORECASE) for p in INSTITUTION_PATTERNS)

def extract_title_block(lines: List[str], max_lines_to_merge: int = 4) -> str:
    block = []

    for line in lines[:8]:
        s = normalize_whitespace(line)
        if not s:
            continue
        if is_table_of_contents_line(s):
            break
        if is_likely_org_line(s):
            break
        if re.search(DATE_PATTERN, s, flags=re.IGNORECASE):
            break
        if extract_contributors_from_line(s):
            break
        if s.lower() in SECTION_LABELS:
            break
        if re.fullmatch(r"\d+", s):
            continue

        # allow short title fragments like "Personal"
        if len(s.split()) <= 5:
            block.append(s)

        if len(block) >= max_lines_to_merge:
            break

    return normalize_whitespace(" ".join(block))

def score_title_candidate(line: str, contributors: List[str]) -> float:
    s = normalize_whitespace(line)

    if is_title_noise(s):
        return -999.0
    if any(s.lower() == c.lower() for c in contributors):
        return -999.0

    score = 0.0
    wc = len(s.split())

    if wc < 2 or wc > 16:
        return -999.0

    if is_likely_org_line(s):
        score -= 3.0

    if 2 <= wc <= 8:
        score += 2.0
    elif 9 <= wc <= 14:
        score += 1.0

    alpha = [ch for ch in s if ch.isalpha()]
    if alpha:
        uppercase_ratio = sum(ch.isupper() for ch in alpha) / len(alpha)
        if uppercase_ratio > 0.85:
            score += 1.2

    if "," in s:
        score -= 2.0

    return score


def extract_title(row: pd.Series, contributors: Optional[List[str]] = None) -> str:
    contributors = contributors or []
    text = safe_string(row.get(NLP_TEXT_COLUMN, "")) or safe_string(row.get(TEXT_COLUMN, ""))
    lines = get_title_page_lines(text, max_lines=20)

    title_block = extract_title_block(lines)
    if len(title_block.split()) >= 2:
        return title_block

    candidates = []
    for line in lines:
        score = score_title_candidate(line, contributors)
        if score > -100:
            candidates.append((line, score))

    if not candidates:
        return ""

    candidates.sort(key=lambda x: x[1], reverse=True)
    best_title, best_score = candidates[0]
    return best_title if best_score >= 1.5 else ""


In [258]:
# Date helpers
def parse_single_date(date_text: str, prefer_day_first: bool = True) -> Optional[datetime]:
    
    date_text = normalize_whitespace(date_text).lower()
    cleaned = clean_date_text(date_text)

    numeric_formats = (
        ["%d-%m-%Y", "%d/%m/%Y", "%d-%m-%y", "%d/%m/%y"]
        if prefer_day_first else
        ["%m-%d-%Y", "%m/%d/%Y", "%m-%d-%y", "%m/%d/%y"]
    )

    other_formats = [
        "%Y-%m-%d",
        "%d %B %Y", "%d %b %Y",
        "%d %B", "%d %b",
        "%B %d %Y", "%b %d %Y",
        "%B %Y", "%b %Y",
        "%Y"
    ]

    for fmt in numeric_formats + other_formats:
        try:
            dt = datetime.strptime(cleaned, fmt)
            if fmt in ("%B %Y", "%b %Y"):
                dt = dt.replace(day=1)
            if fmt == "%Y":
                dt = dt.replace(month=1, day=1)
            return dt
        except ValueError:
            continue

    return None

def parse_date_to_sortable(date_text: str, context_line: str = "") -> Optional[datetime]:
    date_text = normalize_whitespace(date_text)
    context_line = normalize_whitespace(context_line)

    # First try normal European style
    dt_day_first = parse_single_date(date_text, prefer_day_first=True)
    dt_month_first = parse_single_date(date_text, prefer_day_first=False)

    # If only one works, use it
    if dt_day_first and not dt_month_first:
        return dt_day_first
    if dt_month_first and not dt_day_first:
        return dt_month_first

    # If both work, try to infer from a paired range in the same line
    range_matches = re.findall(DATE_PATTERN, context_line, flags=re.IGNORECASE)
    range_matches = [normalize_whitespace(x) for x in range_matches if normalize_whitespace(x)]

    if len(range_matches) >= 2:
        parsed_day_first = [parse_single_date(x, prefer_day_first=True) for x in range_matches[:2]]
        parsed_month_first = [parse_single_date(x, prefer_day_first=False) for x in range_matches[:2]]

        day_first_ok = all(x is not None for x in parsed_day_first)
        month_first_ok = all(x is not None for x in parsed_month_first)

        if day_first_ok and not month_first_ok:
            return dt_day_first
        if month_first_ok and not day_first_ok:
            return dt_month_first

    # fallback preference
    return dt_day_first or dt_month_first

def extract_dates(text: str) -> List[str]:
    text = safe_string(text)
    found = []

    for match in re.findall(DATE_PATTERN, text, flags=re.IGNORECASE):
        clean = normalize_whitespace(match)
        if clean:
            found.append(clean)

    return dedupe_keep_order(found)


def extract_dates_in_order(text: str) -> List[str]:
    return extract_dates(text)

def extract_abstract_text(text: str, max_chars: int = 1600) -> str:
    text = safe_string(text)

    match = re.search(
        r"\babstract\b\s*(.*?)(?=\n\s*(table\s+of\s+contents|\d+(\.\d+)*\s+[A-Za-z]|introduction)\b|\Z)",
        text,
        flags=re.IGNORECASE | re.DOTALL
    )
    if not match:
        return ""

    abstract = match.group(1)
    abstract = re.sub(r"\s+", " ", abstract).strip()
    return abstract[:max_chars]


In [259]:
# Research group classification
def build_research_group_text(
    title: str,
    description: str,
    cleaned_text: str,
    max_chars: int = 2500
) -> str:
    pieces = []

    if title:
        pieces.append(f"Title: {title}")
    if description:
        pieces.append(f"Description: {description}")
    if cleaned_text:
        pieces.append(f"Key content: {cleaned_text[:1200]}")

    text = " ".join(pieces)
    return normalize_whitespace(text)[:max_chars]


def classify_research_group(
    title: str,
    description: str,
    cleaned_text: str,
    candidate_labels: List[str],
    top_k: int = 3
) -> Dict[str, Any]:
    classification_text = build_research_group_text(
        title=title,
        description=description,
        cleaned_text=cleaned_text
    )

    if not classification_text.strip():
        return {
            "research_group": "",
            "research_group_confidence": None,
            "research_group_top3": []
        }

    result = zero_shot_classifier(
        classification_text,
        candidate_labels=candidate_labels,
        multi_label=False,
        hypothesis_template="This document best fits the research group {}."
    )

    labels = result["labels"]
    scores = result["scores"]

    top_items = []
    for label, score in list(zip(labels, scores))[:top_k]:
        top_items.append({
            "label": label,
            "score": round(float(score), 6)
        })

    best_label = labels[0] if labels else ""
    best_score = float(scores[0]) if scores else None

    if best_score is not None and best_score < RESEARCH_GROUP_CONFIDENCE_THRESHOLD:
        best_label = ""

    return {
        "research_group": best_label,
        "research_group_confidence": round(best_score, 6) if best_score is not None else None,
        "research_group_top3": top_items
    }

In [260]:
def extract_body_preview(text: str, max_chars: int = 500) -> str:
    lines = [line.strip() for line in safe_string(text).replace("\r", "\n").splitlines() if line.strip()]

    start_idx = 0
    for i, line in enumerate(lines):
        if re.match(r"^(role and context in the assignment|openminded to different opinions|identifying problems|various solutions|planning & monitoring ahead)\b", line, flags=re.IGNORECASE):
            start_idx = i + 2
            break

    preview = " ".join(lines[start_idx:start_idx + 12])
    return normalize_whitespace(preview)[:max_chars]

In [261]:
# Metadata pipeline
def extract_generic_metadata(row: pd.Series) -> Dict[str, Any]:
    document_id = safe_string(row.get(DOC_ID_COLUMN, ""))
    nlp_text = safe_string(row.get(NLP_TEXT_COLUMN, ""))
    cleaned_text = safe_string(row.get(TEXT_COLUMN, ""))

    working_text = nlp_text or cleaned_text

    contributors = extract_contributors(row)
    title = extract_title(row, contributors=contributors)
    date_info = extract_document_dates(working_text)

    abstract_text = extract_abstract_text(working_text)
    description = abstract_text or extract_body_preview(working_text)

    research_group_info = classify_research_group(
        title=title,
        description=description,
        cleaned_text=cleaned_text,
        candidate_labels=RESEARCH_GROUPS
    )

    return {
        "id": document_id,
        "title": title,
        "title_found": bool(title),
        "contributors": contributors,
        "contributors_found": bool(contributors),
        "start_date": date_info["start_date"],
        "end_date": date_info["end_date"],
        "dates_found": date_info["dates_found"],
        "date_confidence": date_info["date_confidence"],
        "description": description,
        "research_group": research_group_info["research_group"],
        "research_group_confidence": research_group_info["research_group_confidence"],
        "research_group_top3": research_group_info["research_group_top3"],
        "metadata_debug": {
            "front_lines_sample": get_front_lines(working_text)[:12],
            "date_candidates": date_info["date_candidates"][:5]
        }
    }

In [262]:
# Run
def main() -> None:
    df = pd.read_json(INPUT_JSON, orient="records")

    if df.empty:
        raise ValueError("Input JSON is empty.")

    row = df.loc[0]
    result = extract_generic_metadata(row)

    output_dir = Path(OUTPUT_DIR)
    output_dir.mkdir(parents=True, exist_ok=True)

    output_path = output_dir / OUTPUT_FILE
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=4, ensure_ascii=False)

    print(f"Saved mBART metadata output to {output_path}")
    print("\nMETADATA:\n")
    print(json.dumps(result, indent=4, ensure_ascii=False))


if __name__ == "__main__":
    main()

Saved mBART metadata output to ..\..\..\Data\gold\MBART\mbart_metadata_output.json

METADATA:

{
    "id": "doc_01",
    "title": "Personal Professional Development",
    "title_found": true,
    "contributors": [
        "Joeri Harreman"
    ],
    "contributors_found": true,
    "start_date": "26th May 2024",
    "end_date": "",
    "dates_found": [
        "26th May 2024"
    ],
    "date_confidence": 3.9,
    "description": "Collected proof: ..................................................................................................................................................... 4 Openminded to different opinions (2) ......................................................................................................................... 5 Student check: ...........................................................................................................................................................",
    "research_group": "",
    "research_group_confidence": 0.142

In [263]:
# Add this as a temporary debug cell in gold_mbart_metadata.ipynb
df = pd.read_json(INPUT_JSON, orient="records")
row = df.loc[0]

nlp_text = safe_string(row.get(NLP_TEXT_COLUMN, ""))
print("=== RAW NLP_TEXT FRONT (first 500 chars) ===")
print(repr(nlp_text[:500]))

print("\n=== FRONT LINES (raw) ===")
for i, line in enumerate(get_front_lines_raw(nlp_text, max_lines=20)):
    print(f"  [{i}] {repr(line)}")

print("\n=== FRONT LINES (after split_joined) ===")
for i, line in enumerate(get_front_lines(nlp_text, max_lines=20)):
    print(f"  [{i}] {repr(line)}")

print("\n=== CONTRIBUTOR CHECK PER LINE ===")
for line in get_front_lines(nlp_text, max_lines=20):
    names = extract_contributors_from_line(line)
    if names:
        print(f"  LINE: {repr(line)}")
        print(f"  FOUND: {names}")

=== RAW NLP_TEXT FRONT (first 500 chars) ===
'1\n\nPersonal\nProfessional\nDevelopment\n\n26th May 2024\nSubject: PPD\nHand-in portfolio\n\nHBO-ICT\nJoeri Harreman\n\n2\n\nTable of Contents\nRole and context in the assignment (1) ........................................................................................................................ 4\nStudent check: ............................................................................................................................................................ 4\nCollected proof: ..................'

=== FRONT LINES (raw) ===
  [0] '1'
  [1] 'Personal'
  [2] 'Professional'
  [3] 'Development'
  [4] '26th May 2024'
  [5] 'Subject: PPD'
  [6] 'Hand-in portfolio'
  [7] 'HBO-ICT'
  [8] 'Joeri Harreman'
  [9] '2'

=== FRONT LINES (after split_joined) ===
  [0] '1'
  [1] 'Personal'
  [2] 'Professional'
  [3] 'Development'
  [4] '26th May 2024'
  [5] 'Subject: PPD'
  [6] 'Hand-in portfolio'
  [7] 'HBO-ICT'
  [8] 'Joeri Harreman'
  [